# Goodreads Analysis Pipeline (IDS 706 Mini Assignment)

Distributed PySpark pipeline over the UCSD Goodreads Book Graph (6.9 GB Delta, 23.9M rows across books, authors, genres, reviews). Covers transformations (filter, join, groupBy, withColumn, SQL), optimization (early filters, .explain, caching), Parquet output, and an actions-vs-transformations demo.

## Part 0 — Setup and data loading

The companion `Goodreads Books Dataset Ingestion` notebook downloads the UCSD Goodreads Book Graph (~7.5 GB gzipped) and writes four typed Delta tables into `workspace.goodreads`. This pipeline reads those tables and analyzes them.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import time

# Delta tables built by the ingestion notebook (6.91 GB total, 23.9M rows)
books = spark.table("workspace.goodreads.books")
authors = spark.table("workspace.goodreads.authors")
genres = spark.table("workspace.goodreads.book_genres")
reviews = spark.table("workspace.goodreads.reviews")

# Destination for Parquet output (Part 3)
OUTPUT_DIR = "/Volumes/workspace/goodreads/processed"

print(f"books={books.count():,}  authors={authors.count():,}  genres={genres.count():,}  reviews={reviews.count():,}")

books=2,360,655  authors=829,529  genres=5,042,542  reviews=15,739,967


## Part 1 — Research questions (filters, joins, groupBy, withColumn)

1. Do longer reviews correlate with ratings?
2. What do ratings look like for authors with 3+ published books?
3. Which month has the most publications?
4. Do more popular books (higher ratings/review counts) rate higher or lower?
5. How do ratings evolve across a series (3+ books, each with more than the median number of ratings)?
6. What genres do multi-genre authors publish in, and do they tend to switch genres?
7. Which genres have the most publications, highest ratings, most rating variability?
8. Which days/months/years stand out for publications, and when did genres boom?
9. What formats are most popular over the years, and is digital availability growing?
10. How variable are page counts across formats, genres, and decades?
11. Which reviews earn votes, and does review length matter?
12. How concentrated is reader attention across books?

In [0]:
# Q1: Do longer reviews correlate with ratings?
# Early filter (optimization): only rated reviews with text — applied before any aggregation.
review_len = (reviews
    .filter(F.col("rating").between(1, 5) & F.col("review_text").isNotNull())
    .withColumn("review_len", F.length("review_text"))
)

rating_vs_len = review_len.groupBy("rating").agg(
    F.count("*").alias("n_reviews"),
    F.round(F.avg("review_len"), 1).alias("avg_review_len"),
    F.round(F.expr("percentile_approx(review_len, 0.5)"), 1).alias("median_review_len"),
)
display(rating_vs_len.orderBy("rating"))

print("corr(review_len, rating) =", review_len.select(F.round(F.corr("review_len", "rating"), 4)).collect()[0][0])

rating,n_reviews,avg_review_len,median_review_len
1,448016,674.1,294
2,1107096,703.0,352
3,3117525,691.3,349
4,5250205,742.9,368
5,5265240,708.0,331


corr(review_len, rating) = 0.0073


In [0]:
# Q1b: flip the view — average rating per review-length bucket (withColumn + CASE-style buckets)
buckets = ["1: <100", "2: 100-299", "3: 300-999", "4: 1000-2999", "5: 3000+"]
rating_by_bucket = (review_len
    .withColumn("len_bucket",
        F.when(F.col("review_len") < 100, buckets[0])
         .when(F.col("review_len") < 300, buckets[1])
         .when(F.col("review_len") < 1000, buckets[2])
         .when(F.col("review_len") < 3000, buckets[3])
         .otherwise(buckets[4]))
    .groupBy("len_bucket")
    .agg(F.count("*").alias("n_reviews"),
         F.round(F.avg("rating"), 3).alias("avg_rating"))
)
display(rating_by_bucket.orderBy("len_bucket"))

len_bucket,n_reviews,avg_rating
1: <100,2981989,3.95
2: 100-299,3971867,3.886
3: 300-999,4896839,3.879
4: 1000-2999,2776778,3.936
5: 3000+,560609,3.932


In [0]:
# Q2: Ratings for authors with 3+ published books (join books <-> authors on primary author)
# NOTE: each row in `books` is an EDITION; `work_id` groups editions of the same title
# (e.g. Stephen King: 1,873 editions but only 244 distinct works). We count both and
# use DISTINCT WORKS for the 3+ filter, so "3+ books" means 3+ actual titles.
# Optimization: filter books BEFORE the join, and qualify shared column names
# (books and authors both have average_rating / ratings_count).
author_books = (books
    .filter(F.col("ratings_count") >= 50)          # early filter: drop books with too few ratings
    .join(authors, books.primary_author_id == authors.author_id, "inner")
)
per_author = author_books.groupBy("name").agg(
    F.count("*").alias("n_editions"),
    F.approx_count_distinct("work_id").alias("n_works"),
    F.round(F.avg(F.col("books.average_rating")), 3).alias("avg_rating"),
    F.round(F.stddev(F.col("books.average_rating")), 3).alias("rating_stddev"),
)

multi_book_authors = per_author.filter(F.col("n_works") >= 3)
display(multi_book_authors.orderBy(F.desc("n_works")).limit(15))

row = multi_book_authors.agg(
    F.count("*").alias("n_authors"),
    F.round(F.avg("avg_rating"), 3).alias("mean_of_author_avgs"),
).collect()[0]
print(f"Authors with 3+ distinct well-rated works: {row['n_authors']:,}, average author rating: {row['mean_of_author_avgs']}")

name,n_editions,n_works,avg_rating,rating_stddev
Francine Pascal,647,602,3.556,0.201
R.L. Stine,596,466,3.728,0.22
Brian Michael Bendis,451,367,3.844,0.323
Robert Kirkman,488,367,4.167,0.268
Nora Roberts,1030,357,4.012,0.182
Anonymous,697,355,4.159,0.421
Carolyn Keene,391,345,3.843,0.158
Ann M. Martin,461,343,3.676,0.185
Agatha Christie,1515,304,3.891,0.207
James Patterson,1283,288,3.898,0.243


Authors with 3+ distinct well-rated works: 42,045, average author rating: 3.877


In [0]:
# Q3: Which month sees the most publications? (early filter on valid years/months)
pub_month = (books
    .filter(F.col("publication_year").between(1900, 2020) & F.col("publication_month").between(1, 12))
    .groupBy("publication_month")
    .agg(F.count("*").alias("n_books"),
         F.round(F.avg("average_rating"), 3).alias("avg_rating"))
)
display(pub_month.orderBy(F.desc("n_books")))

busiest = pub_month.orderBy(F.desc("n_books")).first()
print(f"Busiest publication month: {busiest['publication_month']} with {busiest['n_books']:,} books")

publication_month,n_books,avg_rating
10,147070,3.895
9,146070,3.878
1,138343,3.844
4,129439,3.868
3,128181,3.872
5,127004,3.869
6,119603,3.872
8,116052,3.87
11,113330,3.896
2,107988,3.864


Busiest publication month: 10 with 147,070 books


In [0]:
# Q4: Do more popular books (more ratings) rate higher or lower? Popularity buckets + correlation.
popularity = (books
    .filter(F.col("ratings_count") >= 50)
    .withColumn("popularity_bucket",
        F.when(F.col("ratings_count") < 100, "1: 50-99")
         .when(F.col("ratings_count") < 1000, "2: 100-999")
         .when(F.col("ratings_count") < 10000, "3: 1k-9,999")
         .when(F.col("ratings_count") < 100000, "4: 10k-99,999")
         .when(F.col("ratings_count") < 1000000, "5: 100k-999,999")
         .otherwise("6: 1M+"))
    .groupBy("popularity_bucket")
    .agg(F.count("*").alias("n_books"),
         F.round(F.avg("average_rating"), 3).alias("avg_rating"),
         F.round(F.stddev("average_rating"), 3).alias("rating_stddev"))
)
display(popularity.orderBy("popularity_bucket"))

print("corr(ratings_count, average_rating) =",
      books.filter(F.col("ratings_count") >= 50)
           .select(F.round(F.corr("ratings_count", "average_rating"), 4)).collect()[0][0])

popularity_bucket,n_books,avg_rating,rating_stddev
1: 50-99,261452,3.874,0.359
2: 100-999,409630,3.905,0.325
"3: 1k-9,999",79895,3.956,0.28
"4: 10k-99,999",10143,4.011,0.251
"5: 100k-999,999",927,4.031,0.229
6: 1M+,54,4.081,0.289


corr(ratings_count, average_rating) = 0.0191


In [0]:
# Q5: How do ratings evolve across a series? (3+ numbered books, each with MORE THAN THE
# MEDIAN number of ratings — a data-driven threshold instead of the fixed cutoff.)
# Parse series name and position from titles like "Dune (Dune Universe, #1)".
series_rating_median = books.agg(
    F.expr("percentile_approx(ratings_count, 0.5)").alias("med")
).collect()[0]["med"]
print(f"Median ratings_count across all books: {series_rating_median}")

series_books = (books
    .filter(F.col("ratings_count") > series_rating_median)
    .withColumn("series_name", F.regexp_extract("title", r"\(([^,]+), #", 1))
    .withColumn("series_pos", F.expr("try_cast(regexp_extract(title, '#([0-9]+)', 1) AS int)"))
    .filter((F.col("series_name") != "") & F.col("series_pos").isNotNull())
)

# Keep only series with 3+ distinct numbered positions
series_sizes = series_books.groupBy("series_name").agg(F.countDistinct("series_pos").alias("n_positions"))
long_series = series_sizes.filter(F.col("n_positions") >= 3).select("series_name")

series_trunc = (series_books
    .join(long_series, "series_name", "inner")
    # Cap position at 10 so long-running series don't dominate the tail
    .withColumn("pos", F.least(F.col("series_pos"), F.lit(10)))
)

series_trajectory = series_trunc.groupBy("pos").agg(
    F.countDistinct("series_name").alias("n_series"),
    F.count("*").alias("n_books"),
    F.round(F.avg("average_rating"), 3).alias("avg_rating"),
)
display(series_trajectory.orderBy("pos"))

first, late = series_trajectory.filter("pos=1").first(), series_trajectory.filter("pos>=7").first()
if first and late:
    print(f"Book #1 avg rating: {first['avg_rating']} vs later books (#7+, capped at 10): {late['avg_rating']}")

Median ratings_count across all books: 20


pos,n_series,n_books,avg_rating
0,1265,2273,3.933
1,18988,48351,3.897
2,18646,35095,4.013
3,18201,30561,4.043
4,10199,16556,4.064
5,6629,10583,4.084
6,4632,7490,4.086
7,3306,5336,4.078
8,2489,3908,4.066
9,1952,3101,4.063


Book #1 avg rating: 3.897 vs later books (#7+, capped at 10): 3.976


In [0]:
# Q5b: First vs middle vs LAST book ratings, broken down by series length.
# Uses the same filtered set as Q5 (3+ numbered books, each above the median ratings threshold).
# Note: "first"/"last" are the lowest/highest OBSERVED numbered positions — a series whose
# book #4 falls below the ratings threshold is analyzed on the books we can see.
w_series = Window.partitionBy("series_name")

series_tagged = (series_books
    .join(series_sizes, "series_name")                     # brings n_positions
    .filter(F.col("n_positions") >= 3)
    .withColumn("last_pos", F.max("series_pos").over(w_series))
    .withColumn("first_pos", F.min("series_pos").over(w_series))
    .withColumn("book_role",
        F.when(F.col("series_pos") == F.col("first_pos"), "1: first")
         .when(F.col("series_pos") == F.col("last_pos"), "3: last")
         .otherwise("2: middle"))
    .withColumn("length_bucket",
        F.when(F.col("n_positions") == 3, "3 books")
         .when(F.col("n_positions") == 4, "4 books")
         .when(F.col("n_positions") == 5, "5 books")
         .when(F.col("n_positions") <= 10, "6-10 books")
         .otherwise("10+ books"))
)

# (a) Overall: first vs middle vs last
display(series_tagged.groupBy("book_role").agg(
    F.count("*").alias("n_books"),
    F.round(F.avg("average_rating"), 3).alias("avg_rating"),
).orderBy("book_role"))

# (b) By series length: does the pattern differ for 3-book vs 5-book vs longer series?
display(series_tagged.groupBy("length_bucket", "book_role").agg(
    F.approx_count_distinct("series_name").alias("n_series"),
    F.round(F.avg("average_rating"), 3).alias("avg_rating"),
).orderBy("length_bucket", "book_role"))

book_role,n_books,avg_rating
1: first,46007,3.894
2: middle,109058,4.028
3: last,27813,4.053


length_bucket,book_role,n_series,avg_rating
10+ books,1: first,1161,3.942
10+ books,2: middle,1161,4.007
10+ books,3: last,1161,4.024
3 books,1: first,9320,3.87
3 books,2: middle,9320,3.995
3 books,3: last,9320,4.028
4 books,1: first,4156,3.88
4 books,2: middle,4156,4.005
4 books,3: last,4156,4.05
5 books,1: first,2134,3.9


In [0]:
# Q6: What genres do multi-genre authors publish in, and do they switch genres?
books_genres = (books
    .join(genres, "book_id", "inner")
    .filter(F.col("genre").isNotNull() & (F.col("genre") != ""))
)

author_genre_sets = books_genres.groupBy("primary_author_id").agg(
    F.collect_set("genre").alias("genres"),
)
multi_genre = author_genre_sets.filter(F.size("genres") > 1)
row = multi_genre.agg(
    F.count("*").alias("n_multi_genre"),
    F.round(F.avg(F.size("genres")), 2).alias("avg_n_genres"),
).collect()[0]
total_authors = author_genre_sets.count()
print(f"Authors with >1 genre: {row['n_multi_genre']:,} of {total_authors:,} ({100*row['n_multi_genre']/total_authors:.1f}%), averaging {row['avg_n_genres']} genres each")
display(multi_genre.orderBy(F.desc(F.size("genres"))).limit(10))

Authors with >1 genre: 247,216 of 389,007 (63.6%), averaging 3.2 genres each


primary_author_id,genres
17055,"List(fiction, mystery, thriller, crime, poetry, history, historical fiction, biography, romance, young-adult, non-fiction, children, comics, graphic, fantasy, paranormal)"
2855221,"List(mystery, thriller, crime, children, fiction, young-adult, fantasy, paranormal, comics, graphic, history, historical fiction, biography, non-fiction, romance, poetry)"
874602,"List(young-adult, fiction, fantasy, paranormal, children, romance, non-fiction, history, historical fiction, biography, poetry, mystery, thriller, crime, comics, graphic)"
32278,"List(fiction, fantasy, paranormal, non-fiction, mystery, thriller, crime, history, historical fiction, biography, poetry, young-adult, romance, comics, graphic, children)"
2976,"List(fiction, romance, fantasy, paranormal, young-adult, history, historical fiction, biography, children, mystery, thriller, crime, poetry, non-fiction, comics, graphic)"
6632,"List(history, historical fiction, biography, non-fiction, mystery, thriller, crime, fiction, fantasy, paranormal, young-adult, children, comics, graphic, poetry, romance)"
13730,"List(children, fiction, fantasy, paranormal, mystery, thriller, crime, young-adult, romance, non-fiction, history, historical fiction, biography, poetry, comics, graphic)"
18540,"List(poetry, children, fiction, non-fiction, comics, graphic, fantasy, paranormal, history, historical fiction, biography, young-adult, mystery, thriller, crime, romance)"
6943,"List(fiction, fantasy, paranormal, non-fiction, young-adult, children, mystery, thriller, crime, comics, graphic, romance, history, historical fiction, biography, poetry)"
1372374,"List(fiction, fantasy, paranormal, young-adult, mystery, thriller, crime, history, historical fiction, biography, children, comics, graphic, non-fiction, romance, poetry)"


In [0]:
# Q6b: Genre-switch trend — for multi-genre authors, compare their EARLIEST genre
# (min publication year) with their LATEST genre (max publication year).
# Trick: min/max over a struct orders by year first, so we get the genre at that year.
author_genre_years = (books_genres
    .filter(F.col("publication_year").isNotNull())
    .groupBy("primary_author_id", "genre")
    .agg(F.min("publication_year").alias("first_year"),
         F.max("publication_year").alias("last_year"))
)

first_last = author_genre_years.groupBy("primary_author_id").agg(
    F.min(F.struct("first_year", "genre")).alias("first"),
    F.max(F.struct("last_year", "genre")).alias("last"),
)
switchers = first_last.filter(F.col("first.genre") != F.col("last.genre"))
n_switchers = switchers.count()
n_multi2 = multi_genre.count()
print(f"Authors who started and ended in different genres: {n_switchers:,} of {n_multi2:,} multi-genre authors ({100*n_switchers/n_multi2:.1f}%)")

top_switches = switchers.groupBy(
    F.col("first.genre").alias("started_in"),
    F.col("last.genre").alias("ended_in"),
).agg(F.count("*").alias("n_authors"))
display(top_switches.orderBy(F.desc("n_authors")).limit(15))

Authors who started and ended in different genres: 201,896 of 247,216 multi-genre authors (81.7%)


started_in,ended_in,n_authors
"history, historical fiction, biography",non-fiction,46265
fiction,romance,16495
fiction,non-fiction,13473
fiction,"mystery, thriller, crime",11089
children,young-adult,10553
"fantasy, paranormal",romance,10308
"fantasy, paranormal",young-adult,9453
fiction,young-adult,8268
children,non-fiction,7658
"comics, graphic",non-fiction,7460


In [0]:
# Q6c: Most popular genre COMBINATIONS, and how they trend across the years.
# For every author, form all unordered pairs of genres they've published in, dated by
# the year their repertoire expanded to include BOTH genres (max of the two
# first-appearance years) — i.e., the year that combination "formed" for that author.
author_genre = (books_genres
    .filter(F.col("publication_year").between(1900, 2020))
    .groupBy("primary_author_id", "genre")
    .agg(F.min("publication_year").alias("genre_first_year"))
)

# Self-join per author; (a.genre < b.genre) gives each unordered pair exactly once
author_pairs = (author_genre.alias("a")
    .join(author_genre.alias("b"),
        (F.col("a.primary_author_id") == F.col("b.primary_author_id"))
        & (F.col("a.genre") < F.col("b.genre")))
    .select(
        F.col("a.primary_author_id").alias("author_id"),
        F.col("a.genre").alias("genre_1"),
        F.col("b.genre").alias("genre_2"),
        F.greatest(F.col("a.genre_first_year"), F.col("b.genre_first_year")).alias("pair_year"),
    )
    .withColumn("decade", (F.floor(F.col("pair_year") / 10) * 10).cast("int"))
)

# (a) Most popular genre combinations overall
pair_counts = author_pairs.groupBy("genre_1", "genre_2").agg(
    F.approx_count_distinct("author_id").alias("n_authors"),
)
display(pair_counts.orderBy(F.desc("n_authors")).limit(15))

# (b) The fastest-forming combination in each decade (window function)
w_dec = Window.partitionBy("decade").orderBy(F.desc("n_new_authors"))
decade_leaders = (author_pairs
    .groupBy("decade", "genre_1", "genre_2")
    .agg(F.approx_count_distinct("author_id").alias("n_new_authors"))
    .withColumn("rank", F.row_number().over(w_dec))
    .filter(F.col("rank") == 1)
    .drop("rank"))
display(decade_leaders.orderBy("decade"))

# (c) Trend of the single most popular combination across decades
top_pair = pair_counts.orderBy(F.desc("n_authors")).limit(1).collect()[0]
print(f"Trend for the most popular combination: {top_pair['genre_1']} + {top_pair['genre_2']}")
(author_pairs
    .filter((F.col("genre_1") == top_pair["genre_1"]) & (F.col("genre_2") == top_pair["genre_2"]))
    .groupBy("decade")
    .agg(F.approx_count_distinct("author_id").alias("n_new_authors"))
    .orderBy("decade")
).show()

genre_1,genre_2,n_authors
"history, historical fiction, biography",non-fiction,90963
fiction,"history, historical fiction, biography",67947
"fantasy, paranormal",fiction,54519
fiction,romance,51317
fiction,"mystery, thriller, crime",50540
fiction,non-fiction,47643
fiction,young-adult,43096
"history, historical fiction, biography","mystery, thriller, crime",32610
children,fiction,31126
"fantasy, paranormal",romance,30271


decade,genre_1,genre_2,n_new_authors
1900,fiction,"history, historical fiction, biography",133
1910,fiction,"history, historical fiction, biography",105
1920,fiction,"history, historical fiction, biography",175
1930,fiction,"history, historical fiction, biography",254
1940,fiction,"history, historical fiction, biography",367
1950,"history, historical fiction, biography",non-fiction,787
1960,"history, historical fiction, biography",non-fiction,1540
1970,"history, historical fiction, biography",non-fiction,2883
1980,"history, historical fiction, biography",non-fiction,5105
1990,"history, historical fiction, biography",non-fiction,14267


Trend for the most popular combination: history, historical fiction, biography + non-fiction
+------+-------------+
|decade|n_new_authors|
+------+-------------+
|  1900|           79|
|  1910|           83|
|  1920|          123|
|  1930|          193|
|  1940|          300|
|  1950|          787|
|  1960|         1540|
|  1970|         2883|
|  1980|         5105|
|  1990|        14267|
|  2000|        35128|
|  2010|        33172|
+------+-------------+



In [0]:
# Q7: Which genres have the most publications, the highest ratings, and the most variability?
genre_stats = (books_genres
    .filter(F.col("ratings_count") >= 50)
    .groupBy("genre")
    .agg(F.count("*").alias("n_books"),
         F.round(F.avg("average_rating"), 3).alias("avg_rating"),
         F.round(F.stddev("average_rating"), 3).alias("rating_stddev"))
)
display(genre_stats.orderBy(F.desc("n_books")).limit(20))

print("Highest-rated genres (min 5,000 books):")
display(genre_stats.filter(F.col("n_books") >= 5000).orderBy(F.desc("avg_rating")).limit(10))

print("Most rating variability (min 5,000 books):")
display(genre_stats.filter(F.col("n_books") >= 5000).orderBy(F.desc("rating_stddev")).limit(10))

genre,n_books,avg_rating,rating_stddev
fiction,556925,3.883,0.323
romance,313315,3.909,0.329
"history, historical fiction, biography",261661,3.893,0.304
"fantasy, paranormal",240186,3.924,0.321
"mystery, thriller, crime",233782,3.889,0.31
non-fiction,193108,3.927,0.317
young-adult,177508,3.929,0.315
children,101247,3.931,0.308
"comics, graphic",75649,3.959,0.343
poetry,30723,3.964,0.326


Highest-rated genres (min 5,000 books):


genre,n_books,avg_rating,rating_stddev
poetry,30723,3.964,0.326
"comics, graphic",75649,3.959,0.343
children,101247,3.931,0.308
young-adult,177508,3.929,0.315
non-fiction,193108,3.927,0.317
"fantasy, paranormal",240186,3.924,0.321
romance,313315,3.909,0.329
"history, historical fiction, biography",261661,3.893,0.304
"mystery, thriller, crime",233782,3.889,0.31
fiction,556925,3.883,0.323


Most rating variability (min 5,000 books):


genre,n_books,avg_rating,rating_stddev
"comics, graphic",75649,3.959,0.343
romance,313315,3.909,0.329
poetry,30723,3.964,0.326
fiction,556925,3.883,0.323
"fantasy, paranormal",240186,3.924,0.321
non-fiction,193108,3.927,0.317
young-adult,177508,3.929,0.315
"mystery, thriller, crime",233782,3.889,0.31
children,101247,3.931,0.308
"history, historical fiction, biography",261661,3.893,0.304


In [0]:
# Q8: Which years stand out for publications, and when did each genre boom?
# (a) Biggest publication years
pub_year = (books
    .filter(F.col("publication_year").between(1900, 2020))
    .groupBy("publication_year")
    .agg(F.count("*").alias("n_books"),
         F.round(F.avg("average_rating"), 3).alias("avg_rating"))
)
display(pub_year.orderBy(F.desc("n_books")).limit(15))

# (b) Peak year per genre — window function ranks each genre's years by book count
books_genres_yr = books_genres.filter(F.col("publication_year").between(1900, 2020))
genre_year = books_genres_yr.groupBy("genre", "publication_year").agg(
    F.count("*").alias("n_books"),
    F.round(F.avg("average_rating"), 3).alias("avg_rating"),
)
w = Window.partitionBy("genre").orderBy(F.desc("n_books"))
genre_peaks = genre_year.withColumn("rank", F.row_number().over(w)).filter(F.col("rank") == 1)
display(genre_peaks.orderBy(F.desc("n_books")).limit(15))

publication_year,n_books,avg_rating
2013,163752,3.883
2014,156410,3.923
2012,151180,3.846
2015,135735,3.942
2016,122417,3.964
2011,116963,3.824
2010,91230,3.814
2009,77452,3.803
2017,73019,4.023
2008,68488,3.79


genre,publication_year,n_books,avg_rating,rank
fiction,2013,88926,3.866,1
romance,2013,58870,3.861,1
"fantasy, paranormal",2013,43114,3.911,1
"mystery, thriller, crime",2013,39611,3.871,1
"history, historical fiction, biography",2012,37607,3.843,1
young-adult,2013,29787,3.922,1
non-fiction,2012,28831,3.864,1
children,2013,14060,3.939,1
"comics, graphic",2015,10889,3.915,1
poetry,2013,4045,4.006,1


In [0]:
# Q9: Book formats over time, and is digital availability growing?
books_fmt = books.withColumn("format_clean", F.lower(F.trim("format")))

# (a) Most common formats overall
display(books_fmt.filter(F.col("format").isNotNull())
    .groupBy("format_clean").agg(F.count("*").alias("n_books"))
    .orderBy(F.desc("n_books")).limit(12))

# (b) Share of books marked as ebooks, by publication year (1990-2020)
digital_share = (books_fmt
    .filter(F.col("publication_year").between(1990, 2020))
    .withColumn("is_digital", (F.lower(F.col("is_ebook")) == "true").cast("int"))
    .groupBy("publication_year")
    .agg(F.count("*").alias("n_books"),
         F.round(F.avg("is_digital") * 100, 1).alias("pct_digital"))
)
display(digital_share.orderBy("publication_year"))

format_clean,n_books
paperback,895986
,646754
hardcover,359897
ebook,188757
kindle edition,125567
mass market paperback,42266
audio cd,30604
audiobook,17400
audible audio,11074
audio,8360


publication_year,n_books,pct_digital
1990,9711,0.5
1991,10482,0.4
1992,11518,0.5
1993,12775,0.5
1994,13743,0.5
1995,15442,0.5
1996,16808,0.7
1997,18823,0.7
1998,21035,0.8
1999,23046,1.0


In [0]:
# Q9b: How many books (editions of the same work) are available in more than one format,
# and are multi-format works increasingly available digitally?
work_formats = (books_fmt
    .filter(F.col("format").isNotNull())
    .groupBy("work_id")
    .agg(F.approx_count_distinct("format_clean").alias("n_formats"),
         F.max(F.lower(F.col("is_ebook"))).alias("has_ebook"))   # 'true' > 'false' lexicographically
)
row = work_formats.agg(
    F.count("*").alias("n_works"),
    F.sum(F.when(F.col("n_formats") > 1, 1).otherwise(0)).alias("multi_format"),
    F.sum(F.when(F.col("has_ebook") == "true", 1).otherwise(0)).alias("has_ebook"),
).collect()[0]
print(f"Works: {row['n_works']:,} | in >1 format: {row['multi_format']:,} ({100*row['multi_format']/row['n_works']:.1f}%) | with an ebook edition: {row['has_ebook']:,} ({100*row['has_ebook']/row['n_works']:.1f}%)")

# Digital availability of multi-format works, by decade of first publication
multi_digital = (books_fmt
    .filter(F.col("publication_year").between(1950, 2020))
    .groupBy("work_id")
    .agg(F.min("publication_year").alias("first_year"),
         F.approx_count_distinct("format_clean").alias("n_formats"),
         F.max(F.lower(F.col("is_ebook"))).alias("has_ebook"))
    .filter(F.col("n_formats") > 1)
    .withColumn("decade", (F.floor(F.col("first_year") / 10) * 10).cast("int"))
)
display(multi_digital.groupBy("decade").agg(
    F.count("*").alias("n_multi_format_works"),
    F.round(F.avg(F.when(F.col("has_ebook") == "true", 1).otherwise(0)) * 100, 1).alias("pct_with_ebook"),
).orderBy("decade"))

Works: 1,521,963 | in >1 format: 272,583 (17.9%) | with an ebook edition: 548,392 (36.0%)


decade,n_multi_format_works,pct_with_ebook
1950,2058,34.7
1960,3627,23.9
1970,5123,19.6
1980,8500,22.5
1990,17532,25.3
2000,43232,38.2
2010,91663,73.7
2020,1,100.0


### Beyond ratings — where the data actually varies

Average ratings cluster in a narrow 3.8-3.9 band for almost every slice. These questions use measures with real spread: page counts (1 → 14,000+), review votes (0 → tens of thousands), and ratings counts (1 → millions).

In [0]:
# Q10: Page counts — far more variable than ratings (num_pages spans ~1 to 14,000+)
pages_df = books.filter(F.col("num_pages") > 0)

pcts = pages_df.agg(F.expr("percentile_approx(num_pages, array(0.10, 0.25, 0.50, 0.75, 0.90, 0.99))").alias("p")).collect()[0]["p"]
print(f"num_pages percentiles [10th, 25th, 50th, 75th, 90th, 99th]: {pcts}")

# (a) Length by format — audiobooks vs board books vs novels
fmt_pages = (pages_df
    .withColumn("format_clean", F.lower(F.trim("format")))
    .filter(F.col("format_clean").isin("paperback", "hardcover", "mass market paperback", "ebook", "kindle edition", "audiobook", "audio cd", "board book", "audible audio"))
    .groupBy("format_clean")
    .agg(F.count("*").alias("n_books"),
         F.round(F.avg("num_pages"), 1).alias("avg_pages"),
         F.expr("percentile_approx(num_pages, 0.5)").alias("median_pages")))
display(fmt_pages.orderBy(F.desc("n_books")))

# (b) Length by genre
display(pages_df.join(genres, "book_id").groupBy("genre").agg(
    F.count("*").alias("n_books"),
    F.round(F.avg("num_pages"), 1).alias("avg_pages"),
).orderBy(F.desc("n_books")).limit(10))

# (c) Are books getting longer? Average pages by decade
display(pages_df.filter(F.col("publication_year").between(1900, 2020))
    .withColumn("decade", (F.floor(F.col("publication_year") / 10) * 10).cast("int"))
    .groupBy("decade").agg(
        F.count("*").alias("n_books"),
        F.round(F.avg("num_pages"), 1).alias("avg_pages"),
    ).orderBy("decade"))

num_pages percentiles [10th, 25th, 50th, 75th, 90th, 99th]: [60, 150, 247, 344, 448, 835]


format_clean,n_books,avg_pages,median_pages
paperback,828132,270.4,252
hardcover,333405,275.7,256
ebook,127772,236.7,219
kindle edition,94488,244.5,228
mass market paperback,37848,320.6,304
audio cd,8497,32.5,9
audiobook,5318,64.3,11
board book,5084,26.3,18
audible audio,4905,34.9,10


genre,n_books,avg_pages
fiction,884620,285.4
"history, historical fiction, biography",481482,313.5
romance,421009,300.5
non-fiction,389375,280.2
"fantasy, paranormal",373828,288.4
"mystery, thriller, crime",360985,312.5
young-adult,267942,270.2
children,193510,167.4
"comics, graphic",124388,212.8
poetry,68230,206.2


decade,n_books,avg_pages
1900,533,267.4
1910,638,321.6
1920,1212,269.6
1930,1666,304.5
1940,2767,266.3
1950,6503,251.1
1960,14985,247.4
1970,26387,244.3
1980,52446,259.5
1990,133040,268.3


In [0]:
# Q11: Review engagement — votes are extremely skewed: most reviews get none, a few get tens of thousands
vp = reviews.agg(F.expr("percentile_approx(n_votes, array(0.50, 0.75, 0.90, 0.99, 0.999))").alias("p")).collect()[0]["p"]
print(f"n_votes percentiles [50th, 75th, 90th, 99th, 99.9th]: {vp}")

# (a) Does writing more earn more votes? Votes by review-length bucket
votes_by_len = (reviews
    .filter(F.col("rating").between(1, 5) & F.col("review_text").isNotNull())
    .withColumn("review_len", F.length("review_text"))
    .withColumn("len_bucket",
        F.when(F.col("review_len") < 100, "1: <100")
         .when(F.col("review_len") < 300, "2: 100-299")
         .when(F.col("review_len") < 1000, "3: 300-999")
         .when(F.col("review_len") < 3000, "4: 1000-2999")
         .otherwise("5: 3000+"))
    .groupBy("len_bucket")
    .agg(F.count("*").alias("n_reviews"),
         F.round(F.avg("n_votes"), 1).alias("avg_votes"),
         F.expr("percentile_approx(n_votes, 0.9)").alias("p90_votes")))
display(votes_by_len.orderBy("len_bucket"))

# (b) The 10 most-upvoted reviews
top_reviews = (reviews.orderBy(F.desc("n_votes")).limit(10)
    .join(books.select("book_id", "title"), "book_id")
    .select("n_votes", "n_comments", "rating", "title", F.substring("review_text", 1, 90).alias("review_snippet")))
display(top_reviews)

n_votes percentiles [50th, 75th, 90th, 99th, 99.9th]: [0, 1, 2, 19, 83]


len_bucket,n_reviews,avg_votes,p90_votes
1: <100,2981989,0.4,1
2: 100-299,3971867,0.5,1
3: 300-999,4896839,0.9,2
4: 1000-2999,2776778,2.4,5
5: 3000+,560609,5.3,11


n_votes,n_comments,rating,title,review_snippet
3167,537,5,The Goldfinch,"So listen. Look. I am a READER, right? I mean, I read all the time, everywhere, every day,"
3942,922,2,"Mockingjay (The Hunger Games, #3)",SPOILERS AHEAD!! What. the. f***. Words can't begin to express my disappointment. I boug
16695,1044,0,"Doors of Stone (The Kingkiller Chronicle, #3)","While it's nice to see folks out there giving this book five stars, and in some cases even"
2642,174,5,"The Lies of Locke Lamora (Gentleman Bastard, #1)","Back when I was first published, people made a lot of comparisons between me and Scott Lyn"
3222,172,5,The Book Thief,"I give this 5 stars, BUT there is a disclaimer: If you want a fast read, this book is not"
2764,290,1,Esio Trot,** spoiler alert ** Fuck this book. Don't read it to your kids. That's the short version
2253,162,0,"A Conjuring of Light (Shades of Magic, #3)","With ACOL hitting shelves this month, I guess it's time to finally mark it as read...I don"
2333,160,4,"Harry Potter and the Cursed Child - Parts One and Two (Harry Potter, #8)","It was alright. It was just alright. I know I gave this 4 stars, but to be honest, J.K."
2549,229,1,"Harry Potter and the Cursed Child - Parts One and Two (Harry Potter, #8)",Ok wow where do I even begin with this. My summarized thoughts are that this was a waste
2419,199,5,Fahrenheit 451,I am in 6th grade. My Language Arts teacher assigns us a book report; tells us we can choo


In [0]:
# Q12: How concentrated is reader attention? Ratings COUNT (popularity) is wildly skewed —
# the average rating is narrow, but how many ratings a book gets spans 1 to millions.
n_rated = books.filter(F.col("ratings_count") > 0).count()
total_ratings = books.agg(F.sum("ratings_count").alias("t")).collect()[0]["t"]
print(f"{n_rated:,} rated books, {total_ratings:,} total ratings")

w_pop = Window.orderBy(F.desc("ratings_count"))
concentration = (books
    .filter(F.col("ratings_count") > 0)
    .withColumn("rn", F.row_number().over(w_pop))
    .withColumn("cum_share", F.round(
        F.sum("ratings_count").over(w_pop.rowsBetween(Window.unboundedPreceding, Window.currentRow))
        / F.lit(total_ratings), 4))
    .filter(F.col("rn").isin([max(1, int(c * n_rated)) for c in [0.001, 0.01, 0.05, 0.10, 0.25, 0.50]]))
    .withColumn("top_pct", F.round(F.col("rn") / F.lit(n_rated), 4))
    .select("top_pct", "cum_share", "ratings_count"))
display(concentration)

# The 10 most-rated books (the attention champions)
display(books.orderBy(F.desc("ratings_count"))
    .select("title", "ratings_count", "average_rating", "publication_year").limit(10))

2,347,259 rated books, 958,938,616 total ratings


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


top_pct,cum_share,ratings_count
0.001,0.4226,45380
0.01,0.6876,4635
0.05,0.8543,733
0.1,0.9106,296
0.25,0.9658,77
0.5,0.9911,20


title,ratings_count,average_rating,publication_year
"The Hunger Games (The Hunger Games, #1)",4899965,4.34,2008
"Harry Potter and the Sorcerer's Stone (Harry Potter, #1)",4765497,4.45,1997
"Twilight (Twilight, #1)",3941381,3.57,2006
To Kill a Mockingbird,3255518,4.26,2006
The Great Gatsby,2758812,3.89,2004
The Fault in Our Stars,2429317,4.26,2012
The Hobbit,2099680,4.25,2002
The Catcher in the Rye,2086945,3.79,2001
Pride and Prejudice,2078406,4.25,2000
"Angels & Demons (Robert Langdon, #1)",2046499,3.86,2006


## Part 2 — SQL queries (spark.sql)

The same DataFrames registered as temporary views, queried with SQL.

In [0]:
# Register temp views for SQL queries
books.createOrReplaceTempView("books")
authors.createOrReplaceTempView("authors")
genres.createOrReplaceTempView("genres")
reviews.createOrReplaceTempView("reviews")
print("temp views registered")

temp views registered


In [0]:
# SQL query 1: top genres by number of well-rated publications, with rating variability
sql_genres = spark.sql("""
    SELECT g.genre,
           COUNT(*) AS n_books,
           ROUND(AVG(b.average_rating), 3) AS avg_rating,
           ROUND(STDDEV(b.average_rating), 3) AS rating_stddev
    FROM books b
    JOIN genres g ON b.book_id = g.book_id
    WHERE b.ratings_count >= 50
    GROUP BY g.genre
    ORDER BY n_books DESC
    LIMIT 10
""")
display(sql_genres)

genre,n_books,avg_rating,rating_stddev
fiction,556925,3.883,0.323
romance,313315,3.909,0.329
"history, historical fiction, biography",261661,3.893,0.304
"fantasy, paranormal",240186,3.924,0.321
"mystery, thriller, crime",233782,3.889,0.31
non-fiction,193108,3.927,0.317
young-adult,177508,3.929,0.315
children,101247,3.931,0.308
"comics, graphic",75649,3.959,0.343
poetry,30723,3.964,0.326


In [0]:
# SQL query 2: highest-rated prolific authors (3+ DISTINCT WORKS — editions of the same
# title are grouped by work_id, so 3+ means 3+ actual books, not 3+ formats)
sql_authors = spark.sql("""
    SELECT a.name,
           COUNT(*) AS n_editions,
           COUNT(DISTINCT b.work_id) AS n_works,
           ROUND(AVG(b.average_rating), 2) AS avg_rating
    FROM books b
    JOIN authors a ON b.primary_author_id = a.author_id
    WHERE b.ratings_count >= 50
    GROUP BY a.name
    HAVING COUNT(DISTINCT b.work_id) >= 3
    ORDER BY avg_rating DESC, n_works DESC
    LIMIT 15
""")
display(sql_authors)

name,n_editions,n_works,avg_rating
Brian Simmons,5,5,4.84
Bianca,18,18,4.82
KC Mills,10,10,4.82
Elias Zapple,3,3,4.81
KB Cole,15,15,4.8
A.O. Esther,7,7,4.8
M. Skye,4,4,4.8
Leesa Abbott,3,3,4.8
Nikki Brown,11,11,4.79
John Fennimore,3,3,4.79


## Part 3 — Optimization and performance analysis

Techniques:
* **Early filters**: filter to well-rated/valid-year books *before* joins and aggregations (Q2, Q6-Q9)
* **Column pruning**: select only needed columns after joins
* **`.explain()`**: inspect the physical plan — filter pushdown into Delta scans, join strategy, shuffle-free aggregations where possible
* **Caching**: materialize a reused result and compare action latency
* **Partitioned Parquet output**: write results partitioned by `publication_year` so downstream reads prune files

In [0]:
# Physical plan of the genre-year aggregation pipeline (join + filter + groupBy)
query_plan = (books
    .filter(F.col("ratings_count") >= 50)   # early filter -> pushed into the Delta scan
    .join(genres, "book_id")
    .filter(F.col("publication_year").between(1900, 2020))
    .groupBy("genre", "publication_year")
    .agg(F.count("*").alias("n_books"), F.avg("average_rating").alias("avg_rating"))
)
query_plan.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonGroupingAgg (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonGroupingAgg (9)
                     +- PhotonProject (8)
                        +- PhotonBroadcastHashJoin Inner (7)
                           :- PhotonShuffleExchangeSource (5)
                           :  +- PhotonShuffleMapStage (4)
                           :     +- PhotonShuffleExchangeSink (3)
                           :        +- PhotonProject (2)
                           :           +- PhotonScan parquet workspace.goodreads.books (1)
                           +- PhotonScan parquet workspace.goodreads.book_genres (6)


(1) PhotonScan parquet workspace.goodreads.books
Output [4]: [book_id#32003, average_rating#32009, ratings_count#32010L, publication_year#3201

In [0]:
# Caching demo: genre_stats is reused across several analyses — cache it and time the difference.
(1) PhotonScan parquet workspace.goodreads.books
Output [4]: [book_id#32003, average_rating#32009, ratings_count#32010L, publication_year#32013]
DictionaryFilters: [(ratings_count#32010L >= 50), ((publication_year#32013 >= 1900) AND (publication_year#32013 <= 2020))]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-cnxzne2ems/uc/d49bc7bf-e5d9-4c8f-88a2-958e1d517f77/ad732559-8f51-4a2a-8572-e7824a1f65b3/__unitystorage/catalogs/9aaa0fb3-529f-4a2e-bffc-cf4ec751e1e3/tables/83deafac-dad7-47fe-9544-8ded27f3457a]
ReadSchema: struct<book_id:string,average_rating:double,ratings_count:bigint,publication_year:int>
RequiredDataFilters: [isnotnull(publication_year#32013), isnotnull(ratings_count#32010L), (publication_year#32013 >= 1900), (publication_year#32013 <= 2020), isnotnull(book_id#32003), (ratings_count#32010L >= 50)]

(2) PhotonProject
Input [4]: [book_id#32003, average_rating#32009, ratings_count#32010L, publication_year#32013]
Arguments: [book_id#32003, average_rating#32009, publication_year#32013]

(3) PhotonShuffleExchangeSink
Input [3]: [book_id#32003, average_rating#32009, publication_year#32013]
Arguments: SinglePartition

(4) PhotonShuffleMapStage
Input [3]: [book_id#32003, average_rating#32009, publication_year#32013]
Arguments: EXECUTOR_BROADCAST, [id=#25844]

(5) PhotonShuffleExchangeSource
Input [3]: [book_id#32003, average_rating#32009, publication_year#32013]
Arguments: false

(6) PhotonScan parquet workspace.goodreads.book_genres
Output [2]: [book_id#32026, genre#32027]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-cnxzne2ems/uc/d49bc7bf-e5d9-4c8f-88a2-958e1d517f77/ad732559-8f51-4a2a-8572-e7824a1f65b3/__unitystorage/catalogs/9aaa0fb3-529f-4a2e-bffc-cf4ec751e1e3/tables/234b40b8-72af-47da-bb58-ad8da32aee65]
OptionalDataFilters: [hashedrelationcontains(book_id#32026)]
ReadSchema: struct<book_id:string,genre:string>
RequiredDataFilters: [isnotnull(book_id#32026)]

(7) PhotonBroadcastHashJoin
Left keys [1]: [book_id#32003]
Right keys [1]: [book_id#32026]
Join type: Inner
Join condition: None

(8) PhotonProject
Input [5]: [book_id#32003, average_rating#32009, publication_year#32013, book_id#32026, genre#32027]
Arguments: [average_rating#32009, publication_year#32013, genre#32027]

(9) PhotonGroupingAgg
Input [3]: [average_rating#32009, publication_year#32013, genre#32027]
Arguments: [genre#32027, publication_year#32013], [partial_count(1) AS count#32054L, partial_avg(average_rating#32009) AS (sum#32057, count#32058L)], [count#32053L, sum#32055, count#32056L], [genre#32027, publication_year#32013, count#32054L, sum#32057, count#32058L], false

(10) PhotonShuffleExchangeSink
Input [5]: [genre#32027, publication_year#32013, count#32054L, sum#32057, count#32058L]
Arguments: hashpartitioning(genre#32027, publication_year#32013, 1024)

(11) PhotonShuffleMapStage
Input [5]: [genre#32027, publication_year#32013, count#32054L, sum#32057, count#32058L]
Arguments: ENSURE_REQUIREMENTS, [id=#25856]

(12) PhotonShuffleExchangeSource
Input [5]: [genre#32027, publication_year#32013, count#32054L, sum#32057, count#32058L]
Arguments: false

(13) PhotonGroupingAgg
Input [5]: [genre#32027, publication_year#32013, count#32054L, sum#32057, count#32058L]
Arguments: [genre#32027, publication_year#32013], [finalmerge_count(merge count#32054L) AS count(1)#32033L, finalmerge_avg(merge sum#32057, count#32058L) AS avg(average_rating)#32034], [count(1)#32033L, avg(average_rating)#32034], [genre#32027, publication_year#32013, count(1)#32033L AS n_books#31981L, avg(average_rating)#32034 AS avg_rating#31982], true

(14) PhotonColumnarToRow
Input [4]: [genre#32027, publication_year#32013, n_books#31981L, avg_rating#31982]

(15) PhotonResultStage
Input [4]: [genre#32027, publication_year#32013, n_books#31981L, avg_rating#31982]

(16) AdaptiveSparkPlan
Output [4]: [genre#32027, publication_year#32013, n_books#31981L, avg_rating#31982]
Arguments: isFinalPlan=false


== Photon Explanation ==
The query is fully supported by Photon.
== Optimizer Statistics (table names per statistics state) ==
  missing = 
  partial = 
  full    = book_genres, books

07:06 PM (10s)
29:
Caching demo
Caching demo
$0
t0 = time.time()
genre_stats.count()          # cold: computes the full join + aggregation
cold_s = time.time() - t0

genre_stats.cache()          # mark for in-memory caching
t0 = time.time()
genre_stats.count()          # builds the cache (still computes)
build_s = time.time() - t0

t0 = time.time()
genre_stats.count()          # warm: served from cache
warm_s = time.time() - t0

print(f"cold: {cold_s:.2f}s | cache build: {build_s:.2f}s | cached: {warm_s:.2f}s | speedup: {cold_s/max(warm_s, 1e-9):.1f}x")
genre_stats.unpersist()      # clean up

cold: 1.10s | cache build: 0.51s | cached: 0.48s | speedup: 2.3x


DataFrame[genre: string, n_books: bigint, avg_rating: double, rating_stddev: double]

In [0]:
# Write pipeline outputs as Parquet; partition the genre-year table by publication_year
# so downstream queries on a year range only read matching files.
genre_year.write.mode("overwrite").partitionBy("publication_year").parquet(f"{OUTPUT_DIR}/genre_year_stats")
per_author.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/author_stats")

# Verify the outputs by reading them back
for name in ["genre_year_stats", "author_stats"]:
    df = spark.read.parquet(f"{OUTPUT_DIR}/{name}")
    print(f"{name}: {df.count():,} rows")

sample = spark.read.parquet(f"{OUTPUT_DIR}/genre_year_stats").filter("publication_year = 2013")
print(f"Reading a single partition (2013): {sample.count():,} rows — file pruning, no full scan")

genre_year_stats: 1,202 rows
author_stats: 150,063 rows
Reading a single partition (2013): 10 rows — file pruning, no full scan


## Part 4 — Actions vs transformations (lazy evaluation demo)

Transformations (`filter`, `select`, `groupBy`, `withColumn`, `join`) build a logical plan and return instantly — **no data is touched**. Actions (`count`, `collect`, `show`, `write`) trigger the actual Spark job.

In [0]:
# Transformations: instant, lazy — nothing executes yet
t0 = time.time()
transformed = (reviews
    .filter("rating = 5")
    .select("review_id", "book_id", "rating")
    .groupBy("book_id")
    .count()
)
define_s = time.time() - t0
print(f"Building 3 transformations + groupBy: {define_s:.6f}s (no Spark job, pure plan construction)")

# Action: now Spark executes the whole plan
t0 = time.time()
n_books = transformed.count()
action_s = time.time() - t0
print(f"count() action: {action_s:.2f}s to scan 15.7M reviews and return {n_books:,} books with 5-star reviews")

Building 3 transformations + groupBy: 0.000593s (no Spark job, pure plan construction)
count() action: 1.10s to scan 15.7M reviews and return 1,023,284 books with 5-star reviews


## Part 5 — Optional: MLlib regression (bonus)

Predict a book's `average_rating` from popularity (log ratings count), engagement (log review count), length (pages), and age — with a predict-the-mean baseline to judge whether the features help at all.

In [0]:
# Optional (bonus): MLlib linear regression — can we predict a book's average rating?
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

ml_data = (books
    .filter(F.col("ratings_count") >= 50)                     # early filters: well-rated books only
    .filter(F.col("num_pages").between(1, 1000))            # cap outliers: 99th pct is 835, and a 14k-page book would wreck a linear model
    .filter(F.col("publication_year").between(1900, 2020))
    .withColumn("log_ratings", F.log10("ratings_count"))       # popularity is log-skewed (see Q12)
    .withColumn("log_reviews", F.log10(F.col("text_reviews_count") + 1))
    .withColumn("book_age", F.lit(2018) - F.col("publication_year"))  # dataset was collected ~2017-18
    .select("average_rating", "log_ratings", "log_reviews", "num_pages", "book_age")
    .na.drop()
)

feature_cols = ["log_ratings", "log_reviews", "num_pages", "book_age"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
ml_vec = assembler.transform(ml_data).select("average_rating", "features")

train, test = ml_vec.randomSplit([0.8, 0.2], seed=42)
print(f"train: {train.count():,} rows | test: {test.count():,} rows")

lr = LinearRegression(featuresCol="features", labelCol="average_rating", regParam=0.1)
model = lr.fit(train)

preds = model.transform(test)
evaluator = RegressionEvaluator(labelCol="average_rating", predictionCol="prediction")
print(f"\nLinearRegression  RMSE: {evaluator.setMetricName('rmse').evaluate(preds):.4f} | R2: {evaluator.setMetricName('r2').evaluate(preds):.4f}")

# Baseline: always predict the mean training rating — does the model beat it?
mean_rating = train.agg(F.avg("average_rating")).collect()[0][0]
baseline = test.withColumn("prediction", F.lit(mean_rating))
print(f"Baseline (mean {mean_rating:.3f}) RMSE: {evaluator.setMetricName('rmse').evaluate(baseline):.4f}")

print("\nCoefficients (all else equal):")
for name, coef in zip(feature_cols, model.coefficients):
    print(f"  {name}: {coef:+.5f}")

train: 413,752 rows | test: 103,546 rows

LinearRegression  RMSE: 0.3201 | R2: 0.0346
Baseline (mean 3.877) RMSE: 0.3258

Coefficients (all else equal):
  log_ratings: +0.06279
  log_reviews: -0.03224
  num_pages: +0.00023
  book_age: +0.00009


In [0]:
# Optional 5b: flip the target — can we predict POPULARITY instead of rating?
# Target: log10(ratings_count). Features: the AUTHOR's track record (prior popularity,
# prior avg rating — leave-one-out so the book's own popularity never leaks in), pages, age.
# One row per WORK (editions of the same title share ratings; we keep the best-known edition per work).
work_lvl = (books
    .filter(F.col("ratings_count") >= 50)
    .filter(F.col("num_pages").between(1, 1000))
    .filter(F.col("publication_year").between(1900, 2020))
    .groupBy("work_id")
    .agg(F.max("ratings_count").alias("ratings_count"),
         F.max("average_rating").alias("average_rating"),
         F.max("num_pages").alias("num_pages"),
         F.min("publication_year").alias("publication_year"),
         F.max("primary_author_id").alias("primary_author_id"))
)

pop = (work_lvl
    .withColumn("log_pop", F.log10("ratings_count"))
    .withColumn("book_age", F.lit(2018) - F.col("publication_year"))
)

# Leave-one-out author aggregates: what did this author's OTHER books do?
auth = pop.groupBy("primary_author_id").agg(
    F.count("*").alias("n_works"),
    F.sum("log_pop").alias("sum_log_pop"),
    F.sum("average_rating").alias("sum_rating"),
)
global_mean_pop = pop.agg(F.avg("log_pop")).collect()[0][0]

pop_feat = (pop.join(auth, "primary_author_id")
    .withColumn("prior_author_pop",
        F.when(F.col("n_works") > 1, (F.col("sum_log_pop") - F.col("log_pop")) / (F.col("n_works") - 1))
         .otherwise(F.lit(global_mean_pop)))            # single-book authors: fall back to the global mean
    .withColumn("prior_author_rating",
        F.when(F.col("n_works") > 1, (F.col("sum_rating") - F.col("average_rating")) / (F.col("n_works") - 1)))
    .select("log_pop", "prior_author_pop", "prior_author_rating", "num_pages", "book_age")
    .na.drop()
)

pop_features = ["prior_author_pop", "prior_author_rating", "num_pages", "book_age"]
pop_vec = VectorAssembler(inputCols=pop_features, outputCol="features").transform(pop_feat).select("log_pop", "features")

ptrain, ptest = pop_vec.randomSplit([0.8, 0.2], seed=42)
print(f"train: {ptrain.count():,} works | test: {ptest.count():,} works")

plr = LinearRegression(featuresCol="features", labelCol="log_pop", regParam=0.1)
pmodel = plr.fit(ptrain)
pop_evaluator = RegressionEvaluator(labelCol="log_pop", predictionCol="prediction")
ppreds = pmodel.transform(ptest)
print(f"\nLinearRegression  RMSE: {pop_evaluator.setMetricName('rmse').evaluate(ppreds):.4f} | R2: {pop_evaluator.setMetricName('r2').evaluate(ppreds):.4f}")

pmean = ptrain.agg(F.avg("log_pop")).collect()[0][0]
pbase = ptest.withColumn("prediction", F.lit(pmean))
print(f"Baseline (mean {pmean:.3f}) RMSE: {pop_evaluator.setMetricName('rmse').evaluate(pbase):.4f}")

print("\nCoefficients (all else equal, target = log10 ratings):")
for name, coef in zip(pop_features, pmodel.coefficients):
    print(f"  {name}: {coef:+.5f}")

train: 246,379 works | test: 61,334 works

LinearRegression  RMSE: 0.5133 | R2: 0.3372
Baseline (mean 2.477) RMSE: 0.6305

Coefficients (all else equal, target = log10 ratings):
  prior_author_pop: +0.63186
  prior_author_rating: +0.07709
  num_pages: +0.00054
  book_age: +0.00528


In [0]:
# Stability check: rerun both regressions over 10 re-randomized 80/20 splits (different seeds).
# If the single-split results were a lucky draw, R2 will swing wildly across seeds.
from statistics import mean as _mean, stdev as _std

ml_eval = RegressionEvaluator(predictionCol="prediction")   # label set per model below

def split_stability(vec_df, label, n_reps=10):
    r2s, rmses, bases = [], [], []
    for seed in range(n_reps):
        tr, te = vec_df.randomSplit([0.8, 0.2], seed=seed)
        m = LinearRegression(featuresCol="features", labelCol=label, regParam=0.1).fit(tr)
        p = m.transform(te)
        r2s.append(ml_eval.setMetricName("r2").setLabelCol(label).evaluate(p))
        rmses.append(ml_eval.setMetricName("rmse").setLabelCol(label).evaluate(p))
        m0 = tr.agg(F.avg(label)).collect()[0][0]
        bases.append(ml_eval.setMetricName("rmse").evaluate(te.withColumn("prediction", F.lit(m0))))
    return r2s, rmses, bases

for name, vec, label in [("5a rating", ml_vec, "average_rating"), ("5b popularity", pop_vec, "log_pop")]:
    r2s, rmses, bases = split_stability(vec, label)
    print(f"{name:14s} R2  {min(r2s):.3f} - {max(r2s):.3f}  (mean {_mean(r2s):.3f} +/- {_std(r2s):.3f})")
    print(f"{'':14s} RMSE {_mean(rmses):.4f} +/- {_std(rmses):.4f}   vs baseline RMSE {_mean(bases):.4f}")

5a rating      R2  0.034 - 0.035  (mean 0.035 +/- 0.000)
               RMSE 0.3216 +/- 0.0004   vs baseline RMSE 0.3274
5b popularity  R2  0.331 - 0.338  (mean 0.334 +/- 0.002)
               RMSE 0.5160 +/- 0.0017   vs baseline RMSE 0.6323


## Findings summary

1. **Review length vs rating**: essentially no correlation (r = 0.007). Very short (<100 chars) and very long (1000+ chars) reviews rate marginally higher (avg 3.93-3.95) than mid-length ones (3.88) — a shallow U-shape.
2. **Prolific authors (3+ distinct works)**: 42,045 authors qualify, averaging 3.877 — counted by distinct `work_id`, since the books table is one row per *edition* (2.36M editions → 1.52M works; Stephen King: 1,873 editions but only 244 works). Most distinct works: Francine Pascal (602), R.L. Stine (466), Agatha Christie (304), James Patterson (288).
3. **Publication month**: October is the busiest month (147,070 books), September a close second; December is the quietest. Average ratings barely differ by month (~3.87).
4. **Popularity vs rating**: more popular books rate slightly HIGHER — avg rating climbs monotonically from 3.874 (50-99 ratings) to 4.081 (1M+), and rating variability shrinks with popularity (stddev 0.36 → 0.23).
5. **Series trajectory** (books with more than the median of 20 ratings): book #1 averages 3.897, ratings RISE through the mid-series (~4.08 around positions 5-7), then dip for very long series (3.976 at position 10+). No evidence of early-series drop-off; the tail sags instead.
   First vs middle vs last book: **3.89 / 4.03 / 4.05** — the first book is always the weakest, and the first→last gap grows with series length (trilogies +0.16, 4-book +0.17, 5-book +0.19, 6-10 book +0.18) except 10+ book series, which flatten (+0.08). Caveat: survivorship bias — only series whose readers keep going produce later books, so "last book rates best" partly reflects who's left rating it.
6. **Multi-genre authors**: 63.6% of authors (247,216) publish in more than one genre (avg 3.2 genres). 81.7% of them start and end their career in different genres; the most common shift is history/historical-fiction/biography → non-fiction (46,265 authors), followed by fiction → romance and fiction → non-fiction. Your "history → historical fiction" hunch is in the same family: non-fiction-ish categories drifting into/beside fiction.
   Most popular genre COMBINATIONS: history/historical-fiction/biography + non-fiction (91.0K authors), fiction + history/historical-fiction (67.9K), fantasy/paranormal + fiction (54.5K), fiction + romance (51.3K). The fastest-forming combo per decade was fiction + history through 1940, then history + non-fiction from 1950-2000 (peaking at 35.1K new authors in the 2000s) — until fantasy/paranormal + fiction took over the 2010s (33.6K): genre-fiction crossovers displaced the literary non-fiction pairing.
7. **Genre stats**: fiction dominates publications (556,925); poetry (3.964) and comics/graphic (3.959) rate highest; plain fiction rates lowest (3.883). Comics have the most rating variability (stddev 0.343), children's books the least (0.304).
8. **Publication booms**: 2013 is the biggest year (163,752 books), with 2012-2016 forming the peak. Every single genre peaked in 2012-2015 — the whole Goodreads catalog boomed together, rather than genre-specific waves.
9. **Formats**: paperback dominates (895,986), then hardcover (359,897), ebook (188,757) + kindle (125,567). Digital share of publications rose from ~0.5% (1990) to ~35% (2013), holding ~26% through 2017. 17.9% of works are available in >1 format; 36% have an ebook edition — and among multi-format works first published in the 2010s, 73.7% have a digital edition, up from 38.2% for the 2000s: digital availability is clearly growing with newer releases.

**Beyond ratings** (average ratings cluster in a narrow 3.8-3.9 band, so these use measures with real spread):

10. **Page counts** (median 247; 10th-90th percentile spans 60 → 448, 99th 835): formats differ dramatically — audiobook "pages" are really track counts (median 9-11), board books median 18, while mass-market paperbacks are the longest print format (avg 321). By genre, history/historical-fiction (314) and mystery/thriller (313) are longest; children (167) and poetry (206) shortest. Books are NOT getting longer: average length has stayed within 244-322 pages across twelve decades.
11. **Review votes** — where engagement actually varies: the median review gets ZERO votes (75th percentile: 1), yet the 99.9th percentile is 83+. Length pays: 3000+ char reviews average 5.3 votes vs 0.4 for <100 chars (13x). The most-upvoted review in the dataset (16,695 votes, 1,044 comments) is a rating-0 snark review of the never-published *Doors of Stone* (Kingkiller #3) — entertaining outrage outperforms praise.
12. **Attention concentration** — popularity is enormously skewed even though ratings are narrow: across 2.35M rated books and 959M total ratings, the top 0.1% of books capture **42.3% of all ratings**, the top 1% capture 68.8%, the top 10% capture 91.1%, and the bottom half of books share under 1%. Champions: The Hunger Games (4.9M ratings), Harry Potter #1 (4.77M), Twilight (3.94M).

**Optional MLlib regression (Part 5)**: a linear regression predicting `average_rating` from log-popularity, log-review-count, pages, and book age (413,752 train / 103,546 test rows) beats the predict-the-mean baseline only slightly — RMSE 0.3201 vs 0.3258, R² 0.035. The largest coefficient is log-popularity (+0.063 per tenfold ratings increase), consistent with finding 4. Takeaway: average ratings are nearly unpredictable from book metadata — quality signal is weak, popularity is winner-take-all.

**Part 5b — flip the target: predict popularity.** Predicting `log10(ratings_count)` from the author's track record (prior popularity and prior avg rating of their *other* works, computed leave-one-out to avoid leakage), pages, and book age works far better: 246,379 train / 61,334 test works, RMSE 0.5133 vs baseline 0.6305, **R² 0.337** — ten times the variance explained by the rating model. The dominant coefficient is prior author popularity (+0.63 per unit of log-popularity): an author's past audience is the single best predictor of a book's audience. Caveat: the split is random, so works by the same author span train and test — the signal we want (author serial correlation) is also why a stricter time- or author-based split would score lower. A stability check (next cell) rerunning both models over 10 re-randomized 80/20 splits confirms this isn't a lucky draw: 5a R² stays in 0.034-0.035, 5b in 0.331-0.338 (RMSE 0.5160 ± 0.0017 vs baseline 0.6323) — the ~10x gap holds across every seed.

**Edition caveat**: the books table has one row per *edition*; `work_id` groups editions of the same title (2.36M editions → 1.52M works, ~1.55 editions/work). Counts labeled "books" in findings 7-8 (genre stats, booms) are edition-level; the author analysis (Q2) counts distinct works.

**Performance**: the `.explain()` output shows filters pushed into the Delta scans (RequiredDataFilters/DictionaryFilters), a broadcast hash join against the small genres table, and two-stage (partial → final) hash aggregation. Caching the reused genre aggregation cut action latency from 1.10s to 0.48s (2.3x). Parquet output is partitioned by publication_year, so reading one year reads 10 rows-worth of files instead of scanning all 1,202 rows.

**Actions vs transformations**: building 3 transformations + a groupBy took 0.0006s (pure plan construction, no Spark job); the first action (count) took 1.10s to scan 15.7M reviews.